In [ ]:
import pandas as pd
df= pd.read_csv("../dataset/cardiovascular/cardiovascular.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.columns)

In [2]:
df.drop(
    columns=[
        "id",
        "age",
        "bp_category",
        "bp_category_encoded"
    ],
    inplace=True
)

df.head()
print(df.columns)

Index(['gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc',
       'smoke', 'alco', 'active', 'cardio', 'age_years', 'bmi'],
      dtype='str')


In [ ]:
print("Height < 100 cm :", (df["height"] < 100).sum())
print("Height > 220 cm :", (df["height"] > 220).sum())
print("Weight < 30 kg :", (df["weight"] < 30).sum())
print("Weight > 200 kg :", (df["weight"] > 200).sum())
print("BMI < 10 :", (df["bmi"] < 10).sum())
print("BMI > 60 :", (df["bmi"] > 60).sum())
print("Systolic < 80 :", (df["ap_hi"] < 80).sum())
print("Systolic > 250 :", (df["ap_hi"] > 250).sum())

print("Diastolic < 40 :", (df["ap_lo"] < 40).sum())
print("Diastolic > 150 :", (df["ap_lo"] > 150).sum())

In [ ]:
df[df["height"] < 100]
df[df["height"] > 220]
df[df["weight"] < 30]
df[df["bmi"] < 10]
df[df["bmi"] > 60]

In [4]:
df = df[
    (df["height"] >= 120) &
    (df["height"] <= 220) &
    (df["bmi"] >= 10) &
    (df["bmi"] <= 60)
]

print(df.shape)

print(df["height"].min())
print(df["height"].max())

print(df["bmi"].min())
print(df["bmi"].max())

(68128, 13)
120
207
10.72664359861592
59.52380952380953


In [ ]:
print(df["cardio"].value_counts())
print(df["cardio"].value_counts(normalize=True) * 100)

In [5]:
print(df["cholesterol"].value_counts().sort_index())

cholesterol
1    51158
2     9183
3     7787
Name: count, dtype: int64


In [7]:
df["gender"].value_counts()

gender
1    44374
2    23754
Name: count, dtype: int64

In [10]:
df["gluc"].value_counts().sort_index()

gluc
1    57955
2     4997
3     5176
Name: count, dtype: int64

In [ ]:
import matplotlib.pyplot as plt

df.hist(figsize=(18, 12), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

numeric_cols = [
    "height",
    "weight",
    "ap_hi",
    "ap_lo",
    "age_years",
    "bmi"
]

plt.figure(figsize=(15, 8))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 3, i)
    df.boxplot(column=col)
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

features = [
    "age_years",
    "bmi",
    "ap_hi",
    "ap_lo",
    "cholesterol",
    "gluc"
]

plt.figure(figsize=(15, 10))

for i, col in enumerate(features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x="cardio", y=col, data=df)
    plt.title(f"{col} vs Cardio")

plt.tight_layout()
plt.show()

In [16]:
X = df.drop("cardio", axis=1)
y = df["cardio"]

print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (68128, 12)
Target Shape   : (68128,)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (54502, 12)
X_test : (13626, 12)
y_train: (54502,)
y_test : (13626,)


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(54502, 12)
(13626, 12)


Evaluation Function

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("\nConfusion Matrix")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report")
    print(classification_report(y_test, y_pred))

In [21]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42)

lr.fit(X_train_scaled, y_train)

evaluate_model(lr, X_test_scaled, y_test)

Accuracy : 0.7253779539116395
Precision: 0.7531806615776081
Recall   : 0.6600267578415341
F1 Score : 0.7035335129139597

Confusion Matrix
[[5444 1455]
 [2287 4440]]

Classification Report
              precision    recall  f1-score   support

           0       0.70      0.79      0.74      6899
           1       0.75      0.66      0.70      6727

    accuracy                           0.73     13626
   macro avg       0.73      0.72      0.72     13626
weighted avg       0.73      0.73      0.72     13626



In [26]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])
scores = cross_val_score(
    lr_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Average Accuracy :", scores.mean())
print("Std Deviation    :", scores.std())

[0.72354323 0.72897402 0.72647879 0.72976147 0.72425688]
Average Accuracy : 0.7266028771223928
Std Deviation    : 0.002468990716213901


In [22]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

evaluate_model(dt, X_test, y_test)

Accuracy : 0.6272567151034787
Precision: 0.6259553653317028
Recall   : 0.6087408949011447
F1 Score : 0.6172281257065341

Confusion Matrix
[[4452 2447]
 [2632 4095]]

Classification Report
              precision    recall  f1-score   support

           0       0.63      0.65      0.64      6899
           1       0.63      0.61      0.62      6727

    accuracy                           0.63     13626
   macro avg       0.63      0.63      0.63     13626
weighted avg       0.63      0.63      0.63     13626



In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

evaluate_model(rf, X_test, y_test)

Accuracy : 0.7076911786290915
Precision: 0.7092739475289811
Recall   : 0.6912442396313364
F1 Score : 0.7001430399759091

Confusion Matrix
[[4993 1906]
 [2077 4650]]

Classification Report
              precision    recall  f1-score   support

           0       0.71      0.72      0.71      6899
           1       0.71      0.69      0.70      6727

    accuracy                           0.71     13626
   macro avg       0.71      0.71      0.71     13626
weighted avg       0.71      0.71      0.71     13626



In [27]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

scores = cross_val_score(
    rf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Average Accuracy :", scores.mean())
print("Std Deviation    :", scores.std())

[0.70915896 0.71370909 0.7088654  0.70877064 0.70253211]
Average Accuracy : 0.7086072406098971
Std Deviation    : 0.0035590385391470712


In [24]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

knn.fit(X_train_scaled, y_train)

evaluate_model(knn, X_test_scaled, y_test)

Accuracy : 0.6903713488918245
Precision: 0.69435833849969
Recall   : 0.6659729448491155
F1 Score : 0.6798694893391001

Confusion Matrix
[[4927 1972]
 [2247 4480]]

Classification Report
              precision    recall  f1-score   support

           0       0.69      0.71      0.70      6899
           1       0.69      0.67      0.68      6727

    accuracy                           0.69     13626
   macro avg       0.69      0.69      0.69     13626
weighted avg       0.69      0.69      0.69     13626



In [25]:
from sklearn.svm import SVC

svm = SVC(random_state=42)

svm.fit(X_train_scaled, y_train)

evaluate_model(svm, X_test_scaled, y_test)

Accuracy : 0.7305885806546308
Precision: 0.7597755865351921
Recall   : 0.6643377434220307
F1 Score : 0.7088587516853041

Confusion Matrix
[[5486 1413]
 [2258 4469]]

Classification Report
              precision    recall  f1-score   support

           0       0.71      0.80      0.75      6899
           1       0.76      0.66      0.71      6727

    accuracy                           0.73     13626
   macro avg       0.73      0.73      0.73     13626
weighted avg       0.73      0.73      0.73     13626



In [28]:
from sklearn.svm import SVC

svc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(random_state=42))
])

scores = cross_val_score(
    svc_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print(scores)
print("Average Accuracy :", scores.mean())
print("Std Deviation    :", scores.std())

[0.73176281 0.73932188 0.73271686 0.73078899 0.72888073]
Average Accuracy : 0.7326942561239509
Std Deviation    : 0.0035487055852396403


Tuning

In [30]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}
grid_lr = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_lr.fit(X, y)
print("Best Parameters:")
print(grid_lr.best_params_)

print("\nBest CV Accuracy:")
print(grid_lr.best_score_)

Best Parameters:
{'model__C': 0.01}

Best CV Accuracy:
0.7268817622004343


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X, y)

print("Best Parameters:")
print(grid_rf.best_params_)

print("\nBest CV Accuracy:")
print(grid_rf.best_score_)

Best Parameters:
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

Best CV Accuracy:
0.7338685572778431


Tuned Random Forest


In [32]:
from sklearn.ensemble import RandomForestClassifier

best_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=1,
    random_state=42
)

best_rf.fit(X_train, y_train)
evaluate_model(best_rf, X_test, y_test)

Accuracy : 0.731982973726699
Precision: 0.7580130894445377
Recall   : 0.6714731678311283
F1 Score : 0.7121236008198013

Confusion Matrix
[[5457 1442]
 [2210 4517]]

Classification Report
              precision    recall  f1-score   support

           0       0.71      0.79      0.75      6899
           1       0.76      0.67      0.71      6727

    accuracy                           0.73     13626
   macro avg       0.73      0.73      0.73     13626
weighted avg       0.73      0.73      0.73     13626



Feature Importance

In [33]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

,Feature,Importance
3,ap_hi,0.428105
4,ap_lo,0.196135
10,age_years,0.121997
5,cholesterol,0.089325
11,bmi,0.061663
2,weight,0.043967
1,height,0.025954
6,gluc,0.012244
9,active,0.007984
0,gender,0.004528


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance["Feature"],
    feature_importance["Importance"]
)

plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")

plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
y_prob = best_rf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

print("AUC Score:", auc_score)
plt.figure(figsize=(8,6))

plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}", linewidth=2)

plt.plot([0,1], [0,1], linestyle="--", color="gray", label="Random Guess")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Tuned Random Forest")

plt.legend()

plt.grid(alpha=0.3)

plt.show()

In [37]:
import joblib

joblib.dump(best_rf, "../models/cardiovascular_model.pkl")

['../models/cardiovascular_model.pkl']